# LRTIA - Simple Perplexity Test

**The simplest possible test:**

If a model leverages long-range coherence AT ALL, then:
- **Intact text** → Lower perplexity (more predictable)
- **Shuffled text** → Higher perplexity (less predictable)

This is the baseline sanity check. If this doesn't work, nothing will.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import re
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
# Use text that BUILDS on itself - technical/narrative content
# Not generic encyclopedia text, but text with real dependencies

PASSAGES = [
    # Technical explanation that builds concepts
    """Let me explain how neural networks learn. First, we initialize the weights randomly. These weights are the parameters that the network will adjust during training. The network takes an input and multiplies it by these weights to produce an output. We then compare this output to the correct answer using a loss function. The loss tells us how wrong the network was. Using calculus, we compute the gradient of this loss with respect to each weight. The gradient points in the direction that would increase the loss, so we move the weights in the opposite direction. This process is called gradient descent. We repeat this process many times with different training examples. Gradually, the weights converge to values that minimize the loss. The network has now learned to map inputs to outputs. This is the essence of backpropagation, the algorithm that makes deep learning possible.""",

    # Narrative with plot development
    """Detective Chen arrived at the crime scene just after midnight. The victim was a wealthy businessman found in his study. There were no signs of forced entry, which meant the killer was someone the victim knew. Chen noticed a half-empty glass of wine on the desk. She bagged it as evidence, suspecting poison. The security footage showed three visitors that evening: the victim's wife, his business partner, and his lawyer. Chen interviewed each of them the next morning. The wife claimed she left at nine. The partner said he arrived at ten. The lawyer insisted he was never there. But the footage told a different story. The lawyer had lied. Chen brought him back for questioning. Under pressure, he confessed to the murder. The victim had discovered the lawyer was embezzling from the company.""",

    # Argument building to conclusion  
    """The evidence for climate change is overwhelming. Global temperatures have risen by 1.1 degrees Celsius since pre-industrial times. This warming correlates precisely with the increase in atmospheric carbon dioxide. Ice cores show that current CO2 levels are higher than at any point in 800,000 years. The physics is well understood: carbon dioxide traps infrared radiation, warming the planet. Climate models predicted this warming decades ago, and their predictions have proven accurate. Sea levels are rising as ice sheets melt. Extreme weather events are becoming more frequent. Species are shifting their ranges toward the poles. Every major scientific organization agrees on these facts. The only remaining debate is about policy responses, not the underlying science. We must act now to reduce emissions before the damage becomes irreversible.""",

    # Recipe with sequential steps
    """Today I will teach you how to make the perfect risotto. Start by heating your chicken stock in a separate pot and keep it warm throughout the cooking process. In your main pan, sauté finely diced onions in butter until they become translucent. Add the arborio rice and toast it for two minutes, stirring constantly. The rice should become slightly translucent at the edges. Now add a splash of white wine and stir until it is absorbed. This is where patience becomes essential. Add the warm stock one ladle at a time, stirring frequently. Wait until each addition is almost fully absorbed before adding more. This gradual process releases the starch from the rice, creating the creamy texture that makes risotto special. After about eighteen minutes, the rice should be al dente. Remove from heat and stir in butter and parmesan cheese. Let it rest for two minutes before serving.""",

    # Historical narrative with cause and effect
    """The fall of the Roman Empire was not a single event but a gradual process spanning centuries. Several factors contributed to Rome's decline. First, the empire had grown too large to govern effectively. Communication across such vast distances was slow, making it difficult to respond to threats. Second, the army increasingly relied on Germanic mercenaries who had little loyalty to Rome. Third, economic troubles led to inflation and decreased trade. The western provinces became impoverished while the eastern empire remained wealthy. Constantine's decision to move the capital to Constantinople accelerated this divide. When Germanic tribes finally sacked Rome in 410 AD, the city had already lost much of its former glory. The last western emperor was deposed in 476 AD, though this date is somewhat arbitrary. The eastern empire, which we call Byzantium, would survive for another thousand years.""",
]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

print(f"Loaded {len(PASSAGES)} passages")
for i, p in enumerate(PASSAGES):
    print(f"  {i+1}. {len(split_sentences(p))} sentences, {len(tokenizer.encode(p))} tokens")

In [ ]:
@torch.no_grad()
def compute_perplexity(text):
    """Compute perplexity of text under the model."""
    tokens = tokenizer.encode(text, return_tensors="pt").to(model.device)
    
    # Get log probabilities
    outputs = model(tokens, labels=tokens)
    loss = outputs.loss.item()  # Cross-entropy loss = avg negative log prob
    
    perplexity = np.exp(loss)
    return perplexity, loss

# Test
ppl, loss = compute_perplexity("The quick brown fox jumps over the lazy dog.")
print(f"Test perplexity: {ppl:.2f}")

In [ ]:
# Compare intact vs shuffled for each passage
results = []

for i, passage in enumerate(tqdm(PASSAGES, desc="Processing")):
    # Intact
    intact_text = " ".join(passage.split())  # Clean whitespace
    intact_ppl, intact_loss = compute_perplexity(intact_text)
    
    results.append({
        "passage_id": i,
        "condition": "intact",
        "perplexity": intact_ppl,
        "loss": intact_loss,
    })
    
    # Multiple shuffled versions
    sentences = split_sentences(passage)
    for seed in range(5):
        rng = random.Random(42 + i * 100 + seed)
        shuffled_sentences = sentences.copy()
        rng.shuffle(shuffled_sentences)
        shuffled_text = " ".join(shuffled_sentences)
        
        shuffled_ppl, shuffled_loss = compute_perplexity(shuffled_text)
        
        results.append({
            "passage_id": i,
            "condition": "shuffled",
            "perplexity": shuffled_ppl,
            "loss": shuffled_loss,
            "seed": seed,
        })

df = pd.DataFrame(results)
print(f"\nCollected {len(df)} measurements")

In [ ]:
print("=" * 70)
print("RESULTS: Perplexity Comparison")
print("=" * 70)

intact_df = df[df['condition'] == 'intact']
shuffled_df = df[df['condition'] == 'shuffled']

print(f"\nIntact perplexity:   {intact_df['perplexity'].mean():.2f} (±{intact_df['perplexity'].std():.2f})")
print(f"Shuffled perplexity: {shuffled_df['perplexity'].mean():.2f} (±{shuffled_df['perplexity'].std():.2f})")

ratio = shuffled_df['perplexity'].mean() / intact_df['perplexity'].mean()
print(f"\nRatio (shuffled/intact): {ratio:.2f}x")

if ratio > 1.1:
    print("\n✓ Shuffled text has HIGHER perplexity (harder to predict)")
    print("  This confirms the model uses document-level coherence!")
elif ratio < 0.9:
    print("\n✗ Unexpected: Shuffled has LOWER perplexity?!")
else:
    print("\n~ No clear difference in perplexity")

In [ ]:
# Per-passage breakdown
print("\n" + "=" * 70)
print("PER-PASSAGE BREAKDOWN")
print("=" * 70)

passage_names = ["Neural Networks", "Detective Story", "Climate Argument", "Risotto Recipe", "Roman Empire"]

for i in range(len(PASSAGES)):
    intact_ppl = df[(df['passage_id']==i) & (df['condition']=='intact')]['perplexity'].values[0]
    shuffled_ppl = df[(df['passage_id']==i) & (df['condition']=='shuffled')]['perplexity'].mean()
    ratio = shuffled_ppl / intact_ppl
    
    print(f"\n{passage_names[i]}:")
    print(f"  Intact:   {intact_ppl:.2f}")
    print(f"  Shuffled: {shuffled_ppl:.2f}")
    print(f"  Ratio:    {ratio:.2f}x {'↑' if ratio > 1 else '↓'}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Overall comparison
ax = axes[0]
conditions = ['Intact', 'Shuffled']
means = [intact_df['perplexity'].mean(), shuffled_df['perplexity'].mean()]
stds = [intact_df['perplexity'].std(), shuffled_df['perplexity'].std()]
colors = ['#2ecc71', '#e74c3c']

bars = ax.bar(conditions, means, color=colors, yerr=stds, capsize=10)
ax.set_ylabel('Perplexity (lower = more predictable)', fontsize=12)
ax.set_title('Overall Perplexity: Intact vs Shuffled', fontsize=14)

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{mean:.1f}', ha='center', va='bottom', fontsize=12)

# Per-passage comparison
ax = axes[1]
x = np.arange(len(PASSAGES))
width = 0.35

intact_ppls = [df[(df['passage_id']==i) & (df['condition']=='intact')]['perplexity'].values[0] for i in range(len(PASSAGES))]
shuffled_ppls = [df[(df['passage_id']==i) & (df['condition']=='shuffled')]['perplexity'].mean() for i in range(len(PASSAGES))]

ax.bar(x - width/2, intact_ppls, width, label='Intact', color='#2ecc71')
ax.bar(x + width/2, shuffled_ppls, width, label='Shuffled', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(['NN', 'Detective', 'Climate', 'Recipe', 'Rome'], fontsize=10)
ax.set_ylabel('Perplexity')
ax.set_title('Per-Passage Comparison')
ax.legend()

plt.tight_layout()
plt.savefig('perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test
from scipy import stats

# Paired comparison: for each passage, compare intact to mean of shuffled versions
intact_vals = [df[(df['passage_id']==i) & (df['condition']=='intact')]['perplexity'].values[0] for i in range(len(PASSAGES))]
shuffled_vals = [df[(df['passage_id']==i) & (df['condition']=='shuffled')]['perplexity'].mean() for i in range(len(PASSAGES))]

t, p = stats.ttest_rel(intact_vals, shuffled_vals)  # Paired t-test
print(f"\nPaired t-test: t={t:.3f}, p={p:.4f}")

# Also try Wilcoxon signed-rank (non-parametric)
w, p_wilcox = stats.wilcoxon(intact_vals, shuffled_vals)
print(f"Wilcoxon test: W={w:.1f}, p={p_wilcox:.4f}")

if p < 0.05:
    direction = "LOWER" if np.mean(intact_vals) < np.mean(shuffled_vals) else "HIGHER"
    print(f"\n*** SIGNIFICANT: Intact perplexity is {direction} than shuffled (p={p:.4f}) ***")

In [ ]:
# Show example of what shuffling does
print("\n" + "=" * 70)
print("EXAMPLE: What shuffling does to the text")
print("=" * 70)

passage = PASSAGES[0]  # Neural networks
sentences = split_sentences(passage)

print("\nORIGINAL (first 3 sentences):")
for s in sentences[:3]:
    print(f"  - {s}")

rng = random.Random(42)
shuffled = sentences.copy()
rng.shuffle(shuffled)

print("\nSHUFFLED (first 3 sentences):")
for s in shuffled[:3]:
    print(f"  - {s}")

print("\nNotice how concepts appear out of order:")
print("  - Terms used before they're defined")
print("  - 'This process' refers to nothing")
print("  - Logical flow is completely broken")

In [ ]:
# Save results
df.to_csv('perplexity_results.csv', index=False)
print("Saved to perplexity_results.csv")

try:
    from google.colab import files
    files.download('perplexity_results.csv')
    files.download('perplexity_comparison.png')
except:
    pass

## Interpretation

**If intact has LOWER perplexity:**
- The model finds coherent text more predictable
- This proves it uses document-level structure
- The original masking approach should work - we just need to refine it

**If perplexities are similar:**
- The model predicts mostly from local context
- Sentence-level structure is sufficient for prediction
- We'd need more extreme interventions (word-level shuffle)

**If shuffled has LOWER perplexity:**
- Something is very wrong with our experimental setup
- Check for bugs in the code